In [17]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

from torchvision import models
from model_evaluation_helpers import Head

In [18]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
[[255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
Using num_workers = 0


In [19]:
def evaluate_model(modelpath, modeltype, verbose=True):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=512, 
        dropout_rate=0.3, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [20]:
def evaluate_multiple_models(modeldict, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype)
        collected_results[modelname] = results
    return collected_results

In [21]:
path = os.path.join(os.path.expanduser("~"), "Downloads", "Model_BCE_Unweighted", "best_model.pth")
convnext_backbone_metrics = evaluate_model(modelpath=path, modeltype="convnext")

Loading from checkpoint, last run epoch was 9


In [22]:
path = os.path.join(os.path.expanduser("~"), "Downloads", "Model_BCE_EfficientNet_Backbone", "best_model.pth")
efficientnet_backbone_metrics = evaluate_model(modelpath=path, modeltype="efficientnet")

Loading from checkpoint, last run epoch was 13


In [23]:
# No shared feature process layer

class MultiHeadEfficientNet_1(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3, model="convnext"):
        super().__init__()
        
        if model=="convnext":
            backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
            in_features = backbone.classifier[2].in_features
        elif model=="efficientnet":
            backbone = models.efficientnet_v2_s(weights="DEFAULT")
            in_features = backbone.classifier[1].in_features
        else: 
            raise Exception(f"Model {model} not recognised ") 
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        
        backbone.classifier[2] = nn.Linear(in_features, 5)
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, num_conditions)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate)
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        #processed_feats = self.shared_feature_processor(backbone_feats)
        #outputs = [head(processed_feats) for head in self.heads]
        return backbone_feats#torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(os.path.expanduser("~"), "Downloads", "Model_No_Feature_Process_Layer", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_1(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
    model="convnext"
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
no_shared_feature_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)


In [24]:
# No separate classification heads

class MultiHeadEfficientNet_2(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3, model="convnext"):
        super().__init__()
        
        if model=="convnext":
            backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
            in_features = backbone.classifier[2].in_features
        elif model=="efficientnet":
            backbone = models.efficientnet_v2_s(weights="DEFAULT")
            in_features = backbone.classifier[1].in_features
        else: 
            raise Exception(f"Model {model} not recognised ") 
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, num_conditions)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate)
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        #outputs = [head(processed_feats) for head in self.heads]
        return processed_feats#torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(os.path.expanduser("~"), "Downloads", "Model_No_Separate_Heads", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_2(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
    model="convnext"
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
no_separate_heads_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)

In [25]:
# Large shared feature layer

class MultiHeadEfficientNet_3(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=1024, dropout_rate=0.3):
        super().__init__()

        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"

        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.BatchNorm1d(hidden_dim//2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim // 2, hidden_dim // 4, dropout_rate)
            for _ in range(num_conditions)
        ])

    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(os.path.expanduser("~"), "Downloads", "Model_Large_Feature_Layer", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_3(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
larger_shared_layer_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)

In [26]:
agg_results = {
            "ConvNext_Backbone": convnext_backbone_metrics, 
            "Efficientnet_Backbone": efficientnet_backbone_metrics, 
            "No_Shared_Feature_Proc": no_shared_feature_proc_ranking_metrics, 
            "No_Separate_Heads": no_separate_heads_proc_ranking_metrics, 
            "Larger_Shared_Feature_Proc": larger_shared_layer_proc_ranking_metrics}

In [27]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name                    Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
--------------------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
ConvNext_Backbone             0.748961           0.750581        0.751623    0.0266081    0.0139284      0.06772         0.931963       0.938249    0.814109    0.814095
Efficientnet_Backbone         0.734476           0.73361         0.743553    0.020748     0.0152514      0.0688328       0.929043       0.935325    0.803621    0.809656
No_Shared_Feature_Proc        0.748042           0.739978        0.761237    0.0258375    0.0139453      0.0669263       0.934167       0.938892    0.809632    0.817572
No_Separate_Heads             0.75353            0.777582        0.733736    0.0232258    0.0117843      0.0656884       0.93539        0.940783    0.82264

In [28]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
--------------------------  ---------  --------  --------  --------  --------
ConvNext_Backbone            0.752816  0.627976  0.747226  0.763072  0.853717
Efficientnet_Backbone        0.739241  0.621538  0.741573  0.74463   0.825397
No_Shared_Feature_Proc       0.758534  0.625369  0.74856   0.776337  0.831409
No_Separate_Heads            0.756455  0.645624  0.750469  0.760832  0.854271
Larger_Shared_Feature_Proc   0.754204  0.638554  0.763987  0.773463  0.836449


In [29]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
--------------------------  ----------------  ---------------  --------------  --------------  --------------
ConvNext_Backbone                   0.726343         0.698675        0.704651        0.779633        0.843602
Efficientnet_Backbone               0.684642         0.721429        0.747004        0.740506        0.774468
No_Shared_Feature_Proc              0.701754         0.688312        0.730337        0.786535        0.792952
No_Separate_Heads                   0.767705         0.688073        0.716846        0.829868        0.885417
Larger_Shared_Feature_Proc          0.711844         0.721088        0.749054        0.782324        0.806306


In [30]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
--------------------------  -------------  ------------  -----------  -----------  -----------
ConvNext_Backbone                0.781293      0.57027      0.795276       0.7472     0.864078
Efficientnet_Backbone            0.803301      0.545946     0.73622        0.7488     0.883495
No_Shared_Feature_Proc           0.825309      0.572973     0.767717       0.7664     0.873786
No_Separate_Heads                0.74553       0.608108     0.787402       0.7024     0.825243
Larger_Shared_Feature_Proc       0.801926      0.572973     0.779528       0.7648     0.868932


In [31]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
--------------------------  ------------  -----------  ----------  ----------  ----------
ConvNext_Backbone               0.935462     0.897058    0.921842    0.922346    0.983107
Efficientnet_Backbone           0.92492      0.898444    0.915733    0.920625    0.985496
No_Shared_Feature_Proc          0.936374     0.909923    0.917746    0.927873    0.978921
No_Separate_Heads               0.931306     0.907916    0.922481    0.931827    0.983419
Larger_Shared_Feature_Proc      0.936327     0.903011    0.926642    0.935584    0.989724
